# **Neural data analysis — SocialPresentation**
### *stimulus responses and single-neuron selectivity*

Works from the presentation-locked tensor written by `processing_main_abf.ipynb`
(`tensor_dff.npy`, `tensor_labels.npy`, `neuron_regions.npy`). Region analysis is
**optional** — if there's no region file (small FOV) everything runs on all neurons.

- **1** — population trace + neuron heatmap per stimulus (all neurons or a region)
- **2** — single-neuron examples with preferential responses

> Decoding analysis lives in `decoding_main.ipynb`.
> Stimuli: Empty (baseline), 3DMouse (fake), Male, Female (social), FamFood, NewFood (food).

## **Section 0 : Load tensor + config**

In [ ]:
import os
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
from neural_analysis import NeuralAnalysis, DECODERS, set_style   # same folder as this notebook
set_style(font="Arial", size=7)   # Arial 7pt + no top/right spines for every figure

# ---- EDIT ----
DATA       = "/Volumes/Akhil Data/Akhil/ProcessedData/IS-2-1L/2026-07-01"
pre_frames = 15        # MUST match the pre_frames used when the tensor was built
fs         = 15.06     # imaging frame rate (Hz)

X      = np.load(os.path.join(DATA, "tensor_zdff.npy"))
labels = np.load(os.path.join(DATA, "tensor_labels.npy"), allow_pickle=True)
try:
    regions = np.load(os.path.join(DATA, "neuron_regions.npy"), allow_pickle=True)
    print("regions loaded — large-FOV mode")
except FileNotFoundError:
    regions = None
    print("no region file — all-neuron mode")

na = NeuralAnalysis(X, labels, regions=regions, fs=fs, pre_frames=pre_frames)
print("tensor:", na.X.shape, "| stimuli:", sorted(set(na.labels)))
print("regions:", na.region_list())
na.normalize(method="baseline_sub")   # tensor is already Z-scored dF/F (Willmore)

`normalize()` ran on init (z-score to the pre-onset baseline). Adjust the
baseline by changing `pre_frames`, or re-run `na.normalize(method=...)`
(`zscore_pre` / `baseline_sub` / `zscore_session`).

## **Section 1 : Population responses + neuron heatmaps**

Panel-E style: top = population mean ΔF/F (±SEM), bottom = per-neuron heatmap
sorted by response. Dashed lines mark onset (0) and — pass `event_times` — the
presentation offset.

In [ ]:
# default column order: empty cage, fake mouse, male, female, familiar food, unfamiliar food
fig = na.plot_population_heatmap(clip = 1.0, figsize= (6,3), dpi = 800, show_trials=True, smooth_sigma=3)   # ~presentation offset (s); adjust to your median
plt.savefig('population_heatmap.svg', bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

# ---- CONFIG ----
STIMULI = ["Male", "Female", "3DMouse", "FamFood"]

COLORS = {
    "Male": "#27AAE1",
    "Female": "#EC008C",
    "3DMouse": "purple",
    "FamFood": "orange",
}

LEGEND_LABELS = {
    "Male": "male",
    "Female": "female",
    "3DMouse": "fake mouse",
    "FamFood": "familiar food",
}

XLABEL = "time from onset (s)"
YLABEL = "Z-scored ΔF/F"

region = None       # e.g. "ACx"
show_sem = True
smooth_sigma = 1
figsize = (2, 1.5)
dpi = 800

# ---- PLOT ----
def _smooth(x):
    return gaussian_filter1d(x, sigma=smooth_sigma) if smooth_sigma > 0 else x

fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

for stim in STIMULI:
    tr = np.where(na.labels == stim)[0]
    if len(tr) == 0:
        continue

    if region is None or not na.has_regions:
        neuron_mask = np.ones(na.n_neurons, bool)
    else:
        neuron_mask = na.regions == region

    m = na.Xz[tr][:, neuron_mask, :].mean(axis=0)   # (n_neurons, n_frames)
    pop = _smooth(m.mean(0))
    color = COLORS[stim]

    ax.plot(na.time, pop, lw=1.2, color=color, label=LEGEND_LABELS[stim])
    if show_sem:
        sem = _smooth(m.std(0) / np.sqrt(max(m.shape[0], 1)))
        ax.fill_between(na.time, pop - sem, pop + sem, color=color, alpha=0.2)

ax.axvline(0, color="k", ls="--", lw=0.8)
ax.axhline(0, color="0.7", lw=0.5)
ax.set(xlabel=XLABEL, ylabel=YLABEL, xlim=(na.time[0], na.time[-1]))
ax.set_ylim(-0.05, 0.1)
ax.legend(fontsize=6, loc="best", frameon=False)
fig.tight_layout()
plt.savefig('population_traces.svg', bbox_inches='tight')

plt.show()

#### Per region *(large FOV only)* — same figure restricted to one cortical area.

In [ ]:
for reg in na.region_list():
    fig = na.plot_population_heatmap(region=reg, clip = 1.0, figsize= (6,3), dpi = 800, show_trials=True, smooth_sigma=3)
    plt.show()

#### Mean activity by region per stimulus
One subplot per stimulus (6 total). Each subplot shows the trial-averaged mean
trace for every cortical region (excluding "none" neurons). Shared y-axis across
all panels so amplitudes are directly comparable.

In [ ]:
from neural_analysis import STIM_ORDER, STIM_LABELS
from scipy.ndimage import gaussian_filter1d

valid_regions = [r for r in na.region_list() if r.lower() != "none"]
time = na.time
cmap = plt.cm.get_cmap("tab10")

fig, axes = plt.subplots(1, 6, figsize=(6, 1.5), dpi=800, sharey=True, sharex=True)

for j, stim in enumerate(STIM_ORDER):
    ax = axes[j]
    trial_mask = na.labels == stim
    for i, reg in enumerate(valid_regions):
        neuron_mask = na.regions == reg
        # (n_trials_for_stim, n_neurons_in_region, n_frames) -> mean over neurons -> mean over trials
        traces = na.X[trial_mask][:, neuron_mask, :]
        mean_trace = traces.mean(axis=1).mean(axis=0)  # mean across neurons, then trials
        mean_trace = gaussian_filter1d(mean_trace, sigma=2)
        ax.plot(time, mean_trace, lw=1.0, color=cmap(i), label=reg if j == 0 else None)
    ax.axvline(0, color="k", ls="--", lw=0.7)
    ax.axhline(0, color="0.7", lw=0.4)
    ax.set_title(STIM_LABELS.get(stim, stim))
    ax.set_xlabel("time from onset (s)", fontsize=6)
    if j == 0:
        ax.set_ylabel("Z-scored ΔF/F")

fig.legend(valid_regions, loc="center right", frameon=False, fontsize=6,
           bbox_to_anchor=(1.06, 0.5))
#fig.suptitle("Mean regional activity per stimulus", fontsize=8, y=1.02)
plt.tight_layout()
plt.show()

## **Section 2 : Single-neuron examples (preferential responses)**

`preferential_neurons` ranks cells by how selectively they fire to a target
stimulus vs the others (single-trial AUC). Swap `target` for any stimulus.

In [ ]:
pref = na.preferential_neurons(target="Male", n=6, window=(0, 4), min_selectivity=8.0)
display(pref)
na.plot_neuron_examples(pref["neuron_id"].tolist()[:3], smooth_sigma=3, dpi=300); plt.show()

In [ ]:
for tgt in ['3DMouse', 'Empty', 'FamFood', 'Female', 'Male', 'NewFood']:
    p = na.preferential_neurons(target=tgt, n=3, window=(0, 4), min_selectivity=4.0)
    if len(p):
        fig = na.plot_neuron_examples(p["neuron_id"].tolist(), smooth_sigma=2, dpi=300)
        plt.suptitle(f"{tgt}-preferential", y=1.05)
        plt.show()